In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import distance_transform_edt, label, binary_fill_holes
from skimage.morphology import skeletonize

# --- FILE PATHS ---
MASKS_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\refined_masks"
CALIBRATION_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\scale_calibration.csv"
OUTPUT_RESULTS_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_results.csv"

# 1. Load image-specific scale calibration dictionary
df_scale = pd.read_csv(CALIBRATION_CSV)
scale_map = {}
for _, row in df_scale.iterrows():
    base_name = os.path.splitext(str(row['image_name']))[0]
    scale_map[base_name] = float(row['nm_per_pixel'])

def process_single_mask(mask_path, nm_per_px):
    """
    Separates independent membrane slices (connected components)
    and computes thickness profiles along their central skeletons.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Binarize and fill small internal voids
    binary_mask = mask > 128
    binary_mask = binary_fill_holes(binary_mask)

    # Step 2: Separate independent membrane components
    labeled_mask, num_features = label(binary_mask)
    component_data = []

    # Step 3: Measure thickness per component
    for comp_id in range(1, num_features + 1):
        component = (labeled_mask == comp_id)
        
        # Exclude tiny artifacts (< 300 pixels)
        if np.sum(component) < 300:
            continue

        # Distance transform & Skeletonization
        dist_transform = distance_transform_edt(component)
        skeleton = skeletonize(component)

        # Local orthogonal thickness = 2 * distance_transform value along skeleton
        thickness_pixels = dist_transform[skeleton] * 2.0
        thickness_nm = thickness_pixels * nm_per_px

        component_data.append({
            "component_id": comp_id,
            "mean_thickness_nm": np.mean(thickness_nm),
            "median_thickness_nm": np.median(thickness_nm),
            "std_thickness_nm": np.std(thickness_nm),
            "min_thickness_nm": np.min(thickness_nm),
            "max_thickness_nm": np.max(thickness_nm),
            "num_points_measured": len(thickness_nm)
        })

    return component_data

# --- BATCH EXECUTION ---
all_results = []

if os.path.exists(MASKS_FOLDER):
    print("=" * 65)
    print(" PROCESSING REFINED MASKS WITH DYNAMIC SCALES ")
    print("=" * 65)

    for filename in sorted(os.listdir(MASKS_FOLDER)):
        if filename.endswith(('.png', '.tif', '.jpg')):
            mask_path = os.path.join(MASKS_FOLDER, filename)
            base_name = os.path.splitext(filename)[0]

            # Match scale factor from Step 1 CSV
            nm_per_px = scale_map.get(base_name, None)
            
            if nm_per_px is None:
                print(f"[Warning] No scale found for {filename}. Skipping.")
                continue

            # Extract patient ID (e.g. '01-24' from '01-24_03_1g')
            patient_id = base_name.split('_')[0]

            # Process mask components
            comp_results = process_single_mask(mask_path, nm_per_px)
            
            for res in comp_results:
                res["patient_id"] = patient_id
                res["image_name"] = filename
                res["resolution_nm_px"] = nm_per_px
                all_results.append(res)
                
            print(f"✓ Processed {filename:<22} | Found {len(comp_results)} membrane component(s)")

    # Export results to CSV
    df_output = pd.DataFrame(all_results)
    df_output.to_csv(OUTPUT_RESULTS_CSV, index=False)
    
    print("=" * 65)
    print(f"SUCCESS: Analysis complete! Results saved to '{OUTPUT_RESULTS_CSV}'.")
    print("=" * 65)
else:
    print(f"[Error] Directory not found: '{MASKS_FOLDER}'")